# Tabular + PhoBERT: End-to-End Fusion

## 1. Objective

Combine the two deep models in one network trained end to end: the MLP with entity embeddings
(notebook 08) reads the structured attributes, PhoBERT (notebook 09) reads the price-redacted text,
and a joint head predicts the log price from both representations. The question is whether the
text adds information that the structured attributes do not already contain.

## 2. Experimental Setup

GPU required (Colab, *Runtime → Change runtime type → T4 GPU*). Prepare the bundle as for notebook 09
(`python scripts/pack_colab.py`, unzip into `MyDrive/261-dl/`); the first cell mounts Drive. The
tabular branch reuses the architecture selected in notebook 08 (read from `mlp.json`); the text
branch starts from the original pretrained PhoBERT, not from the notebook 09 checkpoint, so that
the fusion model never reuses weights early-stopped on another model's validation curve.

In [ ]:
from pathlib import Path
import json
import os
import sys

try:  # Google Colab: work inside the unzipped bundle on Drive so results persist.
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/261-dl/models")
except ImportError:
    pass

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "experiments").is_dir() and (path / "requirements.txt").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from transformers import AutoModel, AutoTokenizer
from experiments.data import load_data, SEED, REPORTS
from experiments.evaluation import select_candidate, inverse_log, record_result, diagnostics, environment
from experiments.deep import TabularEncoder, EmbeddingMLP, set_seed, get_device
from experiments.phobert import MODEL_NAME, load_text, tokenize, make_loader, predict_log, fit
from experiments.fusion import FusionDataset, FusionModel

set_seed(SEED)
DEVICE = get_device()
if DEVICE.type != "cuda":
    print("WARNING: no GPU. Use Colab with Runtime -> Change runtime type -> GPU.")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 300})
parts, manifest = load_data()
train, validation, test = (parts[name] for name in ["train", "validation", "test"])
texts = load_text(parts)
y_train = train.price.to_numpy()
y_validation = validation.price.to_numpy()
log_y_train = np.log1p(y_train)
log_y_validation = np.log1p(y_validation)
MLP = json.loads((REPORTS / "mlp.json").read_text(encoding="utf-8"))["selected_parameters"]
display(pd.Series({"Seed": SEED, "Device": str(DEVICE),
                   "GPU": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none",
                   "Pretrained model": MODEL_NAME, "Split SHA-256": manifest["split_sha256"]},
                  name="Setting").to_frame())
display(pd.Series(environment()["packages"], name="Installed version").to_frame())

## 3. Data Preprocessing

Both inputs are prepared exactly as in their own notebooks, with every statistic fitted on the
training partition only: `TabularEncoder` (log1p, winsorization, median imputation with missingness
flags, standardization, hierarchical category codes) and PhoBERT tokenization of the segmented,
accented, price-redacted text (truncated at 256 tokens, padded per batch).

In [ ]:
assert train.published_at.max() < validation.published_at.min()
assert validation.published_at.max() < test.published_at.min()
assert set(train.group_id).isdisjoint(validation.group_id)
assert set(train.group_id).isdisjoint(test.group_id)
assert set(validation.group_id).isdisjoint(test.group_id)

encoder = TabularEncoder(min_frequency=5).fit(train)          # training rows only
tables = {name: encoder.transform(parts[name]) for name in ["train", "validation", "test"]}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
ids = {name: tokenize(tokenizer, texts[name]) for name in ["train", "validation", "test"]}
print("Dense inputs:", tables["train"][0].shape[1], "| categorical columns:", tables["train"][1].shape[1])

## 4. Model Implementation

    tabular row → embeddings + residual MLP (notebook 08 architecture) → h_tab (512)
    listing text → PhoBERT → mean pooling → Dropout → Linear(768, 64) → GELU → h_text (64)
    [h_tab | h_text] → LayerNorm → Linear(576, 256) → GELU → Dropout → Linear(256, 1) → log1p(price)

**Late vs. early interaction.** Averaging the predictions of two separately trained models can only
add their opinions. Concatenating their hidden representations lets the joint head learn
interactions, e.g. that "mặt tiền" (street frontage) matters more in some districts than in others,
and lets the gradient of the shared loss adapt each branch to what the other one misses.

**Projection to 64 dimensions.** Without it the 768-dimensional text vector would outnumber the
tabular features and the head would at first rely mostly on the noisier branch. The LayerNorm puts
both parts on a comparable scale before the first fused layer.

**Learning rates.** All new layers (tabular branch, projection, joint head) are trained with the
head learning rate; PhoBERT uses the layer-wise decayed rates of notebook 09. The training loop,
schedule, fp16 and early stopping are the ones of notebook 09 (`experiments/phobert.fit`).

In [ ]:
def build_model():
    tabular = EmbeddingMLP(n_dense=tables["train"][0].shape[1], cardinalities=encoder.cardinalities,
                           hidden=MLP["hidden"], blocks=MLP["blocks"], dropout=MLP["dropout"],
                           embedding_dropout=MLP["embedding_dropout"])
    return FusionModel(AutoModel.from_pretrained(MODEL_NAME), tabular, text_dim=64,
                       target_mean=float(log_y_train.mean())).to(DEVICE)

## 5. Hyperparameter Configuration

The PhoBERT settings are those selected in notebook 09 (top LR 5e-5, layer decay 0.85, head LR 1e-3,
batch 32, at most 4 epochs). Only the length of the frozen-encoder warm-up varies. With 300 steps the
encoder is unfrozen while the tabular branch is still far from converged; with 1,300 steps (one
epoch) the tabular branch has learned most of what it can first, so PhoBERT is pushed to learn what
the attributes miss rather than re-learning the same price level from the text.

In [ ]:
SHARED = {"epochs": 4, "batch_size": 32, "accumulation": 1, "weight_decay": 0.01,
          "warmup_ratio": 0.06, "huber_delta": 1.0,
          "base_lr": 5e-5, "head_lr": 1e-3, "layer_decay": 0.85}
candidates = [{**SHARED, "freeze_steps": 300}, {**SHARED, "freeze_steps": 1300}]
display(pd.DataFrame(candidates))

## 6. Model Training

Each candidate restarts from the pretrained weights with the same seed. Validation log-MAE is checked
twice per epoch; the best weights are restored, and the candidate is selected by validation MAE in
VND. The test partition is used once, afterwards.

In [ ]:
pad = tokenizer.pad_token_id
data = {name: FusionDataset(ids[name], *tables[name]) for name in ["validation", "test"]}
train_data = FusionDataset(ids["train"], *tables["train"], target=log_y_train)
validation_loader = make_loader(data["validation"], 128, False, pad)
test_loader = make_loader(data["test"], 128, False, pad)
histories = []

def fit_candidate(parameters):
    set_seed(SEED)
    model = build_model()
    loader = make_loader(train_data, parameters["batch_size"], True, pad, seed=SEED)
    settings = {k: v for k, v in parameters.items() if k != "batch_size"}
    history, best_step = fit(model, loader, validation_loader, log_y_validation, DEVICE,
                             **settings, log_every=200)
    histories.append(history)
    model.best_step_ = best_step
    return model

def predict_vnd(model, loader):
    return inverse_log(predict_log(model, loader, DEVICE), y_train)

model, selected_parameters, tuning_results = select_candidate(
    candidates, fit_candidate, y_validation, lambda m: predict_vnd(m, validation_loader))
selected_index = candidates.index(selected_parameters)
tuning_results["best_step"] = [int(h.index[h.best].max()) for h in histories]
display(tuning_results)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5), layout="constrained")
for index, history in enumerate(histories):
    ax.plot(history.index, history.validation_log_mae, "-" if index == selected_index else ":",
            marker="o", markersize=3, label=f"freeze {candidates[index]['freeze_steps']} steps")
ax.set(xlabel="Optimizer step", ylabel="Validation MAE of log1p(price)", title="Validation curves")
ax.legend()
plt.show()

## 7. Evaluation

Same rule as every notebook: `expm1` back to VND, bounded by the maximum training price; metrics on
the complete partitions.

In [ ]:
validation_prediction = predict_vnd(model, validation_loader)
test_prediction = predict_vnd(model, test_loader)
extra = {"pretrained_model": MODEL_NAME, "tabular_parameters": MLP,
         "best_step": int(model.best_step_), "device": str(DEVICE),
         "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None}
result, evaluation_table = record_result(
    "fusion", manifest, parts, validation_prediction, test_prediction,
    selected_parameters, tuning_results, extra=extra)
display(evaluation_table.style.format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}",
                                      "R2": "{:.4f}", "MAPE": "{:,.2f}"}))
diagnostics("fusion", test.price, test_prediction)

## 8. Results and Discussion

*To be written once the run is complete and compared with notebooks 08 and 09.*